# Piloto Mc-bajo — Coquimbo, 7 días (Hackatón Meta)
**Alcance cerrado:** 1 segmento (Coquimbo-Illapel), 1 ventana (2019-11-01 → 2019-11-08, M6.5 el 2019-11-04), 4 estaciones CSN (CO02, CO03, CO05, CO06), EQTransformer pretrained. Sin entrenar nada.
**Éxito:** recuperar ≥60 de los 178 eventos Riedel ML≥2.5 de la ventana (USGS solo ve 6) con ≥70% de match.
Runtime: Colab GPU T4, <45 min. Ver `pilot/PILOT.md`.


In [ ]:
import torch
assert torch.cuda.is_available(), "Sin GPU: Colab > Entorno de ejecucion > Cambiar tipo > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
!pip -q install seisbench obspy


In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
STATIONS = ["CO02", "CO03", "CO05", "CO06"]  # red C1 (CSN Chile) via EarthScope
T0, T1 = UTCDateTime("2019-11-01"), UTCDateTime("2019-11-08")
client = Client("EARTHSCOPE")
st = client.get_waveforms("C1", ",".join(STATIONS), "", "BH?", T0, T1)
st.merge(fill_value=0); print(st)
st.write("pilot_waveforms.mseed", format="MSEED")


In [ ]:
import seisbench.models as sbm
model = sbm.EQTransformer.from_pretrained("original")
model.cuda()
# API SeisBench: classify() devuelve picks (P/S) por estacion; annotate() es la alternativa.
try:
    out = model.classify(st)
    picks = out.picks
    print(f"picks totales: {len(picks)}")
    picks.to_csv("pilot_picks.csv", index=False) if hasattr(picks, "to_csv") else None
except Exception as e:
    print("classify no disponible, usando annotate:", e)
    ann = model.annotate(st)
    ann.write("pilot_annotated.mseed", format="MSEED")
    print("anotacion guardada:", len(ann), "trazas")


## Asociación simple y comparación (sin magnitudes nuevas)
Agrupa picks P de ≥3 estaciones dentro de 10 s como evento candidato. Compara contra USGS (FDSN evento) en la misma caja/ventana y contra la verdad Riedel esperada (178 ML≥2.5, medida localmente).

In [ ]:
import urllib.request, json
# USGS en la caja/ventana (lo que ve el modelo vigente)
url = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson"
       "&starttime=2019-11-01&endtime=2019-11-08&minlatitude=-32.2&maxlatitude=-29"
       "&minlongitude=-73&maxlongitude=-69.5&minmagnitude=2.5")
usgs = json.loads(urllib.request.urlopen(url, timeout=30).read())["features"]
print(f"USGS en ventana: {len(usgs)} eventos (esperado ~6 con Mw>=2.5 homogeneizado)")
print("Meta piloto: >=60 eventos asociados y >=70% match Riedel/USGS. Si no, piloto = NULO y no se escala.")


## Entregable demo
Guarda `mini_catalogo_piloto.csv` (tiempo, n_estaciones, picks) y la figura antes/después. Sube ambos a la presentación: el contraste es 6 (USGS) vs ~178 (denso Riedel) en la misma ventana.